## Combine simple classification and NN model

Also move from per-cell to per-patient classification

In [ ]:
import os
import numpy as np
import pandas as pd
import polars as pl

from MLstatkit import Bootstrapping
from sklearn.metrics import roc_auc_score
from sklearn.metrics import class_likelihood_ratios, accuracy_score
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
import yaml

from locpix_points.scripts import evaluate_ensemble

In [ ]:
def feat_extract(features, feature_str, file_filter):
    df_filtered = features.filter(pl.col("file").is_in(file_filter))
    # order in same order as file filter
    # map value to position
    order_map = {v: i for i, v in enumerate(file_filter)}
    
    df_filtered = (
        df_filtered.with_columns(
            pl.col("file").map_dict(order_map).alias("_order")
        )
        .sort("_order")
        .drop("_order")
    )
    return df_filtered["file"], df_filtered[feature_str].to_numpy(), df_filtered["gt"].to_numpy()

def feat_extract_overall(features, feature_list, train_folds, val_folds, test_folds):
    train_list = []
    train_list_gt = []

    val_list = []
    val_list_gt = []

    test_list = []
    test_list_gt = []

    for fold in range(5):
        train_files = train_folds[fold]
        val_files = val_folds[fold]
        test_files = test_folds[fold]

        new_train_files_list, X_train, Y_train = feat_extract(features, feature_list, train_files)
        new_val_files_list, X_val, Y_val = feat_extract(features, feature_list, val_files)
        new_test_files_list, X_test, Y_test = feat_extract(features, feature_list, test_files)

        assert (train_files == new_train_files_list).all()
        assert (val_files == new_val_files_list).all()
        assert (test_files == new_test_files_list).all()

        train_list.append(X_train)
        train_list_gt.append(Y_train)

        val_list.append(X_val)
        val_list_gt.append(Y_val)

        test_list.append(X_test)
        test_list_gt.append(Y_test)

    return train_list, train_list_gt, val_list, val_list_gt, test_list, test_list_gt
    
def simple_classify(model, X_train, Y_train, X_test, Y_test, train_scores, final_probs, final_preds, test_scores_auroc, test_scores_acc, test_scores_plr, feature_names):

    scaler = StandardScaler()
    scaler.fit(X_train)
    X_train = scaler.transform(X_train)
    X_test = scaler.transform(X_test)

    model.fit(X_train, Y_train)

    train_probs = model.predict_proba(X_train)[:,1]
    train_scores.append(roc_auc_score(Y_train, train_probs))

    test_probs = model.predict_proba(X_test)[:,1]
    final_probs.extend(test_probs)
    test_preds = model.predict(X_test)
    final_preds.extend(test_preds)

    test_scores_auroc.append(roc_auc_score(Y_test, test_probs))
    test_scores_acc.append(accuracy_score(Y_test, test_preds))
    test_scores_plr.append(class_likelihood_ratios(Y_test, test_preds)[0])

    return test_probs


In [ ]:
# Preprocessing
input_folder = "../cells/gt_label"
files = os.listdir(input_folder)
channels = ['ereg']

# Class names
class_0_str = "no-response"
class_1_str = "any-response"

# load in splits
k_fold_file = "../output/config/k_fold.yaml"

# load yaml
with open(k_fold_file, "r") as ymlfile:
    k_fold_file = yaml.safe_load(ymlfile)

train_folds = k_fold_file["splits"]["train"]
val_folds = k_fold_file["splits"]["val"]
test_folds = k_fold_file["splits"]["test"]

balanced_accuracy = True

if balanced_accuracy:
    from sklearn.metrics import balanced_accuracy_score as accuracy_score
else:
    from sklearn.metrics import accuracy_score

## Logistic regression classification (repeat of per-cell logistic regression results in other notebook)

c=100, l1_ratio= .25, max_iter= 5000 penalty=elasticnet

In [ ]:
features_path = "../output/simple_classification/features.parquet"
features = pl.read_parquet(features_path)

In [ ]:
features_list = [
    "cell_rgyration",
    "cell_linearity",
    "cell_planarity",
    "cell_length",
    "cell_area",
    "cell_perimeter",
    ]
#
X_train_list, Y_train_list, X_val_list, Y_val_list, X_test_list, Y_test_list = feat_extract_overall(features, features_list, train_folds, val_folds, test_folds)

In [ ]:
# for split in splits
train_scores_simple = []

test_scores_simple_auroc = []
test_scores_simple_acc = []
test_scores_simple_plr = []

final_test_files = []

final_preds_simple = []

simple_feat_importances_list = []

final_probs_simple = []

for fold in range(5):

    for file in test_folds[fold]:
        # evaluate on test set
        final_test_files.append(file)

    X_train = X_train_list[fold]
    Y_train = Y_train_list[fold]

    X_val = X_val_list[fold]
    Y_val = Y_val_list[fold]

    X_train = np.vstack((X_train, X_val))
    Y_train = np.concatenate((Y_train, Y_val))

    X_test = X_test_list[fold]
    Y_test = Y_test_list[fold]

    # simple regression model      
    model = LogisticRegression(C=100, l1_ratio=0.25, max_iter=5000, penalty="elasticnet", solver="saga")

    test_probs = simple_classify(model, X_train, Y_train, X_test, Y_test, train_scores_simple, final_probs_simple, final_preds_simple, test_scores_simple_auroc, test_scores_simple_acc, test_scores_simple_plr, features_list)
    
print("Mean AUROC on test set")
print(np.mean(test_scores_simple_auroc))

print("Mean acc on test set")
print(np.mean(test_scores_simple_acc))


## Best NN model

In [ ]:
nn_final_test_file_list, nn_final_test_probs, nn_final_test_predictions, nn_final_test_gt = evaluate_ensemble.main([
                "-i",
                "../output",
                "-n",
                "nn_model.pt",
                "-m",
                "../config/linked_files.csv",
                "-c",
                "config",
                "-fo",
                "5"
                ])

## Combine probs and calculate metrics

In [ ]:
final_test_files = [x.removesuffix(".parquet") for x in final_test_files] 

In [ ]:
nn_df = pd.DataFrame({"File": nn_final_test_file_list,
                      "nn_prob": nn_final_test_probs,
                     "GT": nn_final_test_gt})

simple_df = pd.DataFrame({"File": final_test_files,
                      "simple_prob": final_probs_simple})

merge_df = result = pd.merge(nn_df, simple_df, on="File")

In [ ]:
combined_probs = np.mean(np.array([merge_df["simple_prob"], merge_df["nn_prob"]]), axis=0)

In [ ]:
df = pd.DataFrame({"File": merge_df["File"], 
                   "GT": merge_df["GT"], 
                   "simple_prob": merge_df["simple_prob"], 
                   "NN_prob": merge_df["nn_prob"], 
                   "Combined_prob": combined_probs})

In [ ]:
simple_rocs = []
NN_rocs = []
comb_rocs = []

simple_accs = []
NN_accs = []
comb_accs = []

for fold in range(5):
    test_fold_files = test_folds[fold]
    test_fold_files = [x.rstrip(".parquet") for x in test_fold_files]
    fold_df = df[df["File"].isin(test_fold_files)]
    
    simple_roc = roc_auc_score(fold_df["GT"], fold_df["simple_prob"])
    NN_roc = roc_auc_score(fold_df["GT"], fold_df["NN_prob"])
    comb_roc = roc_auc_score(fold_df["GT"], fold_df["Combined_prob"])

    simple_acc = accuracy_score(fold_df["GT"], np.where(fold_df["simple_prob"] > 0.5, 1, 0))
    NN_acc = accuracy_score(fold_df["GT"], np.where(fold_df["NN_prob"] > 0.5, 1, 0))
    comb_acc = accuracy_score(fold_df["GT"], np.where(fold_df["Combined_prob"] > 0.5, 1, 0))

    simple_rocs.append(simple_roc)
    NN_rocs.append(NN_roc)
    comb_rocs.append(comb_roc)

    simple_accs.append(simple_acc)
    NN_accs.append(NN_acc)
    comb_accs.append(comb_acc)

print("Mean simple ROC")
print(np.mean(simple_rocs))
print(np.std(simple_rocs, ddof=1))
print("------------\n")

print("Mean NN ROC")
print(np.mean(NN_rocs))
print(np.std(NN_rocs, ddof=1))
print("------------\n")

print("Mean combined ROC")
print(np.mean(comb_rocs))
print(np.std(comb_rocs, ddof=1))
print("------------\n")

print("Mean simple acc")
print(np.mean(simple_accs))
print(np.std(simple_accs, ddof=1))
print("------------\n")

print("Mean NN acc")
print(np.mean(NN_accs))
print(np.std(NN_accs, ddof=1))
print("------------\n")

print("Mean combined acc")
print(np.mean(comb_accs))
print(np.std(comb_accs, ddof=1))
print("------------\n")
    

## Aggregate per patient

In [ ]:
import re
patients = []
files = df["File"]
for file in files:
    match = re.search(r'PAT(\d+)', file)
    p_number = match.group(1) if match else None
    patients.append(p_number)
df["patient"] = patients

print(df.columns)
agg_df = df[["GT", "simple_prob", "NN_prob", "Combined_prob", "patient"]].groupby("patient").mean()
target = agg_df["GT"]

# Simple
simple_prob = agg_df["simple_prob"]
simple_pred = [1 if x > 0.5 else 0 for x in simple_prob]
print("Simple")
print("AUROC: ", roc_auc_score(target, simple_prob))
print("BA : ", accuracy_score(target, simple_pred))

# NN
nn_prob = agg_df["NN_prob"]
nn_pred = [1 if x > 0.5 else 0 for x in nn_prob]
print("NN")
print("AUROC: ", roc_auc_score(target, nn_prob))
print("BA : ", accuracy_score(target, nn_pred))

# Combined
comb_prob = agg_df["Combined_prob"]
comb_pred = [1 if x > 0.5 else 0 for x in comb_prob]
print("Combined")
print("AUROC: ", roc_auc_score(target, comb_prob))
print("BA : ", accuracy_score(target, comb_pred))


In [ ]:
# Loop through multiple metrics
for score in ['roc_auc', 'recall']:
    original_score, conf_lower, conf_upper = Bootstrapping(target, simple_prob, score, threshold=0.5, n_bootstraps=1000)
    print(f"{score.upper()} original score: {original_score:.3f}, confidence interval: [{conf_lower:.3f} - {conf_upper:.3f}]")

    original_score, conf_lower, conf_upper = Bootstrapping(target, nn_prob, score, threshold=0.5, n_bootstraps=1000)
    print(f"{score.upper()} original score: {original_score:.3f}, confidence interval: [{conf_lower:.3f} - {conf_upper:.3f}]")

    original_score, conf_lower, conf_upper = Bootstrapping(target, comb_prob, score, threshold=0.5, n_bootstraps=1000)
    print(f"{score.upper()} original score: {original_score:.3f}, confidence interval: [{conf_lower:.3f} - {conf_upper:.3f}]")

In [ ]:
# save df
output_folder = "../output/combined_classification"
if not os.path.exists(output_folder):
    os.makedirs(output_folder)
df.to_csv(os.path.join(output_folder, "k_fold_df.csv"))